# Calidad, Limpieza y Preparación de Datos#
En este notebook aplicamos las transformaciones necesarias para corregir las anomalías detectadas en la inspección inicial. Cada decisión está respaldada por evidencia y queda registrada de forma trazable en el log del proyecto.

In [34]:
import pandas as pd
import numpy as np

df = pd.read_json("../data/raw/streaming_users_dirty.json")

df_clean = df.copy()

Se crea una copia del dataset original para preservar la información sin modificar los datos crudos

In [35]:
print(df.columns)

Index(['user_id', 'age', 'subscription_plan', 'monthly_watch_time_mins',
       'country', 'favorite_genre', 'last_login_date',
       'customer_support_tickets'],
      dtype='str')


tratamiento de duplicados 

In [36]:
print("Duplicados antes de eliminar:", df_clean.duplicated().sum())

df_clean = df_clean.drop_duplicates()

print("Duplicados después de eliminar:", df_clean.duplicated().sum())

Duplicados antes de eliminar: 126
Duplicados después de eliminar: 0


Se detectaron 126 registros duplicados y fueron eliminados para evitar sesgos en el análisis

correccion de edades invalidas 

In [37]:
df["age"].describe()

count    8160.000000
mean       34.096814
std        14.511304
min        -5.000000
25%        25.000000
50%        33.000000
75%        42.000000
max       150.000000
Name: age, dtype: float64

In [38]:
df.loc[(df["age"] < 0) | (df["age"] > 100), "age"] = np.nan

In [27]:
df["age"].describe()

count    8086.000000
mean       33.522755
std        11.745557
min         0.000000
25%        25.000000
50%        33.000000
75%        41.000000
max        80.000000
Name: age, dtype: float64

correcciones de tiempo de reproduccion invalidas 

In [39]:
df["monthly_watch_time_mins"].describe()

count     7967.000000
mean      1107.346153
std       5310.442604
min       -120.000000
25%        489.200000
50%        757.400000
75%       1045.700000
max      99999.000000
Name: monthly_watch_time_mins, dtype: float64

In [40]:
df.loc[df["monthly_watch_time_mins"] < 0, "monthly_watch_time_mins"] = np.nan

In [41]:
df["monthly_watch_time_mins"].describe()

count     7918.000000
mean      1114.640919
std       5326.036711
min          0.000000
25%        493.000000
50%        760.500000
75%       1047.200000
max      99999.000000
Name: monthly_watch_time_mins, dtype: float64

Durante la etapa de inspección inicial del dataset se detectaron valores inconsistentes en las variables numéricas, Por este motivo, se decidió:

Reemplazar los valores inválidos por NaN (valores nulos)
Tratar estos casos como datos erróneos para no afectar los análisis estadísticos y visualizaciones posteriores

Esta decisión permite mejorar la calidad del dataset, evitar sesgos en los resultados y asegurar un análisis más confiable en la etapa de exploración de datos.

 LIMPIEZA DE TEXTO

subscription_plan

In [47]:
df_clean["subscription_plan"] = df_clean["subscription_plan"].str.strip().str.lower()

df_clean["subscription_plan"] = df_clean["subscription_plan"].replace({
    "basic": "Básico",
    "basico": "Básico",
    "premium": "Premium",
    "premiun": "Premium",
    "pro": "Pro"
})

In [49]:
df_clean["subscription_plan"].value_counts()

subscription_plan
Básico      3609
standard    2833
Premium     1592
Name: count, dtype: int64

LIMPIEZA DE favorite_genre

In [43]:
# [CELDA DE LIMPIEZA]

# A. Forzamos minúsculas y sacamos espacios para que coincida con el diccionario
df_clean["favorite_genre"] = (
    df_clean["favorite_genre"].astype(str).str.lower().str.strip()
)

# B. El diccionario con las claves SÍ O SÍ en minúsculas
mapeo_definitivo = {
    "crime": "Crimen",
    "crimen": "Crimen",
    "thriller": "Thriller",
    "thriiler": "Thriller",
    "thrileer": "Thriller",
    "drama": "Drama",
    "accion": "Acción",
    "acción": "Acción",
    "action": "Acción",
    "romance": "Romance",
    "romantico": "Romance",
    "romántico": "Romance",
    "comedia": "Comedia",
    "comedy": "Comedia",
    "documental": "Documental",
    "documentary": "Documental",
    "doc": "Documental",
    "ciencia ficcion": "Ciencia Ficción",
    "ciencia ficción": "Ciencia Ficción",
    "sci-fi": "Ciencia Ficción",
    "scifi": "Ciencia Ficción",
    "terror": "Terror",
    "horror": "Terror",
}

# C. Aplicamos con .map() y lo guardamos en la misma columna
df_clean["favorite_genre"] = df_clean["favorite_genre"].map(mapeo_definitivo)

# D. Control de calidad: corré esto para verificar que en esta celda ya de bien
print(df_clean["favorite_genre"].unique())

<ArrowStringArray>
[    'Crimen',   'Thriller',      'Drama',     'Acción',    'Romance',
    'Comedia', 'Documental',          nan]
Length: 8, dtype: str


limpieza de contry

In [44]:
# Eliminar espacios al inicio y final
df["country"] = df["country"].str.strip()

# Estandarizar países
df["country"] = df["country"].replace({
    "ARG": "Argentina",
    "argentina": "Argentina",

    "BRA": "Brasil",
    "Brazil": "Brasil",
    "brasil": "Brasil",

    "CHL": "Chile",
    "chile": "Chile",

    "COL": "Colombia",
    "colombia": "Colombia",

    "MEX": "México",
    "Mexico": "México",
    "méxico": "México",

    "PER": "Perú",
    "Peru": "Perú",
    "perú": "Perú",

    "URY": "Uruguay",
    "uruguay": "Uruguay"
})

In [45]:
print(sorted(df["country"].unique()))

['Argentina', 'Brasil', 'Chile', 'Colombia', 'México', 'Perú', 'Uruguay']


Se realizó una limpieza de las variables categóricas para corregir inconsistencias en los datos.

Valores nulos (análisis)

In [13]:
print(df_clean.isnull().sum())
print((df_clean.isnull().sum() / len(df_clean)) * 100)

user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins     193
country                       0
favorite_genre              240
last_login_date             320
customer_support_tickets      0
dtype: int64
user_id                     0.000000
age                         0.000000
subscription_plan           0.000000
monthly_watch_time_mins     2.402290
country                     0.000000
favorite_genre              2.987304
last_login_date             3.983072
customer_support_tickets    0.000000
dtype: float64


Se analizaron valores nulos por columna. Se identificaron variables con baja y media presencia de datos faltantes, lo que permitió definir estrategias de imputación.

tratamiento de nulos

In [14]:
# NUMÉRICA (edad)
df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())

# NUMÉRICA (tiempo de uso)
df_clean["monthly_watch_time_mins"] = df_clean["monthly_watch_time_mins"].fillna(
    df_clean["monthly_watch_time_mins"].median()
)

# CATEGÓRICA (país)
df_clean["country"] = df_clean["country"].fillna(
    df_clean["country"].mode()[0]
)

# CATEGÓRICA (género favorito)
df_clean["favorite_genre"] = df_clean["favorite_genre"].fillna(
    df_clean["favorite_genre"].mode()[0]
)

# CATEGÓRICA (fecha - si tiene nulos)
df_clean["last_login_date"] = df_clean["last_login_date"].fillna(
    df_clean["last_login_date"].mode()[0]
)
df_clean["last_login_date"] = pd.to_datetime(df_clean["last_login_date"], errors="coerce")

df_clean["last_login_date"] = df_clean["last_login_date"].fillna(
    df_clean["last_login_date"].mode()[0]
)

Las variables numéricas fueron imputadas con la mediana para evitar el efecto de outliers, mientras que las variables categóricas fueron completadas con la moda.

In [18]:
print("Nulos finales:")
print(df_clean.isnull().sum())

print("Duplicados finales:", df_clean.duplicated().sum())

Nulos finales:
user_id                     0
age                         0
subscription_plan           0
monthly_watch_time_mins     0
country                     0
favorite_genre              0
last_login_date             0
customer_support_tickets    0
dtype: int64
Duplicados finales: 0


. Dataset procesado

. LOG ETL 

In [52]:
import os
import pandas as pd

# =========================
# 1. CREAR CARPETA LOGS
# =========================
os.makedirs("../logs", exist_ok=True)

# =========================
# 2. LOG ETL
# =========================
log = pd.DataFrame({
    "Paso": [1, 2],
    "Descripción": [
        "Dataset original cargado",
        "Tratamiento de valores nulos y duplicados"
    ],
    "Filas": [len(df), len(df_clean)],
    "Nulos": [
        df.isnull().sum().sum(),
        df_clean.isnull().sum().sum()
    ],
    "Retención (%)": [
        100,
        round(len(df_clean) / len(df) * 100, 2)
    ]
})

# =========================
# 3. GUARDAR LOG (OBLIGATORIO)
# =========================
log.to_csv("../logs/pipeline_log.csv", index=False)

print(log)

# =========================
# 4. GUARDAR DATASET LIMPIO
# =========================
os.makedirs("../data/processed", exist_ok=True)

df_clean.to_csv("../data/processed/dataset_limpio.csv", index=False)

   Paso                                Descripción  Filas  Nulos  \
0     1                   Dataset original cargado   8160    876   
1     2  Tratamiento de valores nulos y duplicados   8034    759   

   Retención (%)  
0         100.00  
1          98.46  


Se implementó un log ETL para registrar el proceso de transformación del dataset, permitiendo comparar el estado inicial y final en términos de filas, valores nulos y retención de datos.